# Experiment 2 - Overview of Deep Learning Libraries

**Aim:** Study the major Python deep learning frameworks (Theano, TensorFlow,
Keras, and PyTorch) and compare them by execution model, abstraction level,
hardware support, and status.

**Approach:** The notebook combines a written framework comparison with a
small, reproducible side-by-side implementation of the same Iris classifier in
Keras/TensorFlow and PyTorch. A short TensorFlow `GradientTape` section shows
what the high-level APIs automate. A guarded PyTensor section demonstrates the
symbolic-graph lineage of Theano without requiring it.

**Dataset:** Iris CSV from a public URL:
`https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv`.
The UCI Iris URL is used as a fallback. No local dataset is used.

**Expected result:** both implementations should classify the three Iris
species with high accuracy (typically 93-100% on the test split). The main
outcome of this experiment is the comparison, not a single accuracy number.

**Runtime:** about 10-20 seconds on CPU.

## 1. Framework comparison

| Library | Developer | Computation graph | Abstraction | Hardware | Status |
|---|---|---|---|---|---|
| Theano | MILA, University of Montreal | Static symbolic | Low level | CPU / GPU | Discontinued after v1.0 (2017); historical |
| TensorFlow | Google | Static graph plus eager execution | Low level and high level via Keras | CPU / GPU / TPU | Active, production-focused |
| Keras | Francois Chollet / Google | Backend-dependent | High level | CPU / GPU / TPU | Active; Keras 3 supports JAX, TensorFlow, and PyTorch backends |
| PyTorch | Meta | Dynamic (define-by-run) | Low level and high level | CPU / GPU | Active, research-dominant |

Theano pioneered symbolic differentiation and GPU acceleration for neural
networks. TensorFlow made computational graphs production-ready and added
eager execution in TF2. Keras provides a compact high-level API. PyTorch made
dynamic graphs and Pythonic debugging the default for research. Theano itself
is no longer maintained, but its ideas live on in Aesara/PyTensor and in the
other frameworks.

## 2. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"TensorFlow : {tf.__version__}")
print(f"PyTorch    : {torch.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Torch device: {DEVICE}")
print(f"TF GPU devices: {tf.config.list_physical_devices('GPU')}")

## 3. Data loading from URL

The primary URL points to the Iris CSV mirrored by the seaborn-data
repository. If it is unavailable, the notebook falls back to the canonical
UCI Iris file and assigns column names manually. Both paths produce the same
four features and three species.

In [ ]:
IRIS_URLS = [
    (
        "seaborn-data mirror",
        "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv",
    ),
    (
        "UCI Iris",
        "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data",
    ),
]
FEATURE_COLUMNS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
REQUIRED_COLUMNS = set(FEATURE_COLUMNS + ["species"])


def load_iris_dataframe():
    errors = []
    for name, url in IRIS_URLS:
        try:
            if name == "UCI Iris":
                frame = pd.read_csv(url, header=None, names=FEATURE_COLUMNS + ["species"])
                frame = frame.dropna().reset_index(drop=True)
            else:
                frame = pd.read_csv(url)
            if not REQUIRED_COLUMNS.issubset(frame.columns):
                raise ValueError(f"Missing columns: {REQUIRED_COLUMNS - set(frame.columns)}")
            print(f"Loaded {len(frame)} rows from {name}")
            print(f"URL: {url}")
            return frame
        except Exception as exc:
            errors.append(f"{name}: {exc}")
    raise RuntimeError("Could not load Iris from any URL:\n" + "\n".join(errors))


iris_df = load_iris_dataframe()
iris_df = iris_df[FEATURE_COLUMNS + ["species"]].copy()
iris_df["species"] = iris_df["species"].astype(str)

print("\nFirst five rows:")
print(iris_df.head().to_string(index=False))
print("\nClass counts:")
print(iris_df["species"].value_counts().to_string())
print("\nSummary statistics:")
print(iris_df.describe().T.to_string())

assert len(iris_df) == 150
assert iris_df["species"].nunique() == 3

## 4. Exploratory plot

In [ ]:
COLORS = {"setosa": "tab:red", "versicolor": "tab:green", "virginica": "tab:blue"}

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for species, group in iris_df.groupby("species"):
    ax.scatter(
        group["petal_length"],
        group["petal_width"],
        label=species,
        color=COLORS.get(species, "gray"),
        alpha=0.8,
        edgecolor="k",
    )
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_title("Iris species separate clearly in petal measurements")
ax.legend(title="Species")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp02_iris_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Preprocessing

The data is split into 70% train, 15% validation, and 15% test with
stratification. Standardization statistics are learned on the training split
only, which avoids leaking test information. The same scaled arrays are used
by both frameworks so the comparison is fair.

In [ ]:
CLASS_NAMES = sorted(iris_df["species"].unique())
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

X = iris_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y = iris_df["species"].map(CLASS_TO_INDEX).to_numpy(dtype=np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

FEATURE_MEAN = X_train.mean(axis=0)
FEATURE_STD = X_train.std(axis=0)
FEATURE_STD[FEATURE_STD == 0.0] = 1.0

X_train = (X_train - FEATURE_MEAN) / FEATURE_STD
X_val = (X_val - FEATURE_MEAN) / FEATURE_STD
X_test = (X_test - FEATURE_MEAN) / FEATURE_STD

print("Class names:", CLASS_NAMES)
print(f"Train: {X_train.shape}, validation: {X_val.shape}, test: {X_test.shape}")
print("Train class counts:", np.bincount(y_train).tolist())
print("Validation class counts:", np.bincount(y_val).tolist())
print("Test class counts:", np.bincount(y_test).tolist())

assert len(CLASS_NAMES) == 3
assert X_train.shape[1] == len(FEATURE_COLUMNS)

## 6. Keras/TensorFlow implementation

Keras is the high-level API shipped with TensorFlow. The model is a small
multilayer perceptron: 4 inputs -> 16 ReLU units -> 3 softmax outputs.

In [ ]:
def build_keras_model(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train.shape[1],), name="features"),
            tf.keras.layers.Dense(16, activation="relu", name="hidden"),
            tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="species"),
        ],
        name="keras_iris_mlp",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


keras_model = build_keras_model()
keras_model.summary()

In [ ]:
keras_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=0,
    )
]

start_time = time.perf_counter()
keras_history = keras_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=16,
    verbose=0,
    callbacks=keras_callbacks,
)
keras_seconds = time.perf_counter() - start_time

keras_test_loss, keras_test_accuracy = keras_model.evaluate(X_test, y_test, verbose=0)
print(f"Keras training time : {keras_seconds:.2f} seconds")
print(f"Keras test loss     : {keras_test_loss:.4f}")
print(f"Keras test accuracy : {keras_test_accuracy:.4f}")

In [ ]:
keras_test_probabilities = keras_model.predict(X_test, verbose=0)
keras_test_predictions = keras_test_probabilities.argmax(axis=1)

print("Keras classification report:")
print(
    classification_report(
        y_test,
        keras_test_predictions,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

## 7. What the high-level API hides: TensorFlow GradientTape

TensorFlow executes eagerly by default. `tf.GradientTape` records operations
and returns their gradients, which is exactly the machinery Keras uses inside
`fit`. The cell below computes one gradient on the full training split without
updating any weights.

In [ ]:
X_train_tf = tf.constant(X_train, dtype=tf.float32)
y_train_tf = tf.constant(y_train, dtype=tf.int32)

W = tf.Variable(tf.random.normal((X_train.shape[1], len(CLASS_NAMES)), stddev=0.1, seed=SEED))
b = tf.Variable(tf.zeros((len(CLASS_NAMES),), dtype=tf.float32))

with tf.GradientTape() as tape:
    logits = tf.matmul(X_train_tf, W) + b
    loss = tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_train_tf, logits=logits)
    )

gradients = tape.gradient(loss, [W, b])
print(f"Toy TensorFlow loss: {loss.numpy():.4f}")
print(f"Gradient shape for W: {gradients[0].shape}")
print(f"Gradient shape for b: {gradients[1].shape}")

## 8. PyTorch implementation

PyTorch builds the graph dynamically during the forward pass. The training
loop below is explicit: zero the gradients, compute the loss, call
`loss.backward()`, and step the optimizer. Early stopping uses the validation
loss and restores the best parameters.

In [ ]:
class IrisNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes),
        )

    def forward(self, x):
        return self.layers(x)


def train_torch_model(
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=200,
    batch_size=16,
    patience=20,
):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = IrisNet(X_train.shape[1], len(CLASS_NAMES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_X = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    val_y = torch.tensor(y_val, dtype=torch.long).to(DEVICE)

    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)

        train_loss = running_loss / total
        train_accuracy = correct / total

        model.eval()
        with torch.no_grad():
            val_logits = model(val_X)
            val_loss = criterion(val_logits, val_y).item()
            val_accuracy = (val_logits.argmax(dim=1) == val_y).float().mean().item()

        history["loss"].append(train_loss)
        history["accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

In [ ]:
start_time = time.perf_counter()
torch_model, torch_history = train_torch_model(
    X_train,
    y_train,
    X_val,
    y_val,
)
torch_seconds = time.perf_counter() - start_time

torch_model.eval()
with torch.no_grad():
    test_logits = torch_model(torch.tensor(X_test, dtype=torch.float32).to(DEVICE))
    torch_test_predictions = test_logits.argmax(dim=1).cpu().numpy()
    torch_test_loss = nn.CrossEntropyLoss()(
        test_logits,
        torch.tensor(y_test, dtype=torch.long).to(DEVICE),
    ).item()

torch_test_accuracy = accuracy_score(y_test, torch_test_predictions)
print(f"PyTorch training time : {torch_seconds:.2f} seconds")
print(f"PyTorch test loss     : {torch_test_loss:.4f}")
print(f"PyTorch test accuracy : {torch_test_accuracy:.4f}")
print()
print("PyTorch classification report:")
print(
    classification_report(
        y_test,
        torch_test_predictions,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

## 9. Training curves for both frameworks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(keras_history.history["loss"], label="Keras train", linewidth=2)
axes[0].plot(keras_history.history["val_loss"], label="Keras validation", linewidth=2)
axes[0].plot(torch_history["loss"], label="PyTorch train", linestyle="--")
axes[0].plot(torch_history["val_loss"], label="PyTorch validation", linestyle="--")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(keras_history.history["accuracy"], label="Keras train", linewidth=2)
axes[1].plot(keras_history.history["val_accuracy"], label="Keras validation", linewidth=2)
axes[1].plot(torch_history["accuracy"], label="PyTorch train", linestyle="--")
axes[1].plot(torch_history["val_accuracy"], label="PyTorch validation", linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp02_framework_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Side-by-side results

In [ ]:
results = pd.DataFrame(
    [
        {
            "Framework": "Keras / TensorFlow",
            "Test loss": keras_test_loss,
            "Test accuracy": keras_test_accuracy,
            "Training seconds": keras_seconds,
        },
        {
            "Framework": "PyTorch",
            "Test loss": torch_test_loss,
            "Test accuracy": torch_test_accuracy,
            "Training seconds": torch_seconds,
        },
    ]
)
print(results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
results.to_csv(OUTPUT_DIR / "exp02_framework_results.csv", index=False)

## 11. Inference with both models

The helper functions accept raw measurements in centimetres. They apply the
training-set standardization internally, so the caller does not need to know
the scaling parameters.

In [ ]:
def keras_predict_flower(features):
    raw = np.asarray(features, dtype=np.float32).reshape(1, -1)
    if raw.shape[1] != len(FEATURE_COLUMNS):
        raise ValueError(f"Expected {len(FEATURE_COLUMNS)} features, got {raw.shape[1]}")
    scaled = (raw - FEATURE_MEAN) / FEATURE_STD
    probabilities = keras_model.predict(scaled, verbose=0)[0]
    index = int(np.argmax(probabilities))
    return CLASS_NAMES[index], float(probabilities[index]), probabilities


def torch_predict_flower(features):
    raw = np.asarray(features, dtype=np.float32).reshape(1, -1)
    if raw.shape[1] != len(FEATURE_COLUMNS):
        raise ValueError(f"Expected {len(FEATURE_COLUMNS)} features, got {raw.shape[1]}")
    scaled = (raw - FEATURE_MEAN) / FEATURE_STD
    torch_model.eval()
    with torch.no_grad():
        logits = torch_model(torch.tensor(scaled, dtype=torch.float32).to(DEVICE))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
    index = int(np.argmax(probabilities))
    return CLASS_NAMES[index], float(probabilities[index]), probabilities


raw_test_sample = X_test[0] * FEATURE_STD + FEATURE_MEAN
true_species = CLASS_NAMES[int(y_test[0])]

print("Raw measurements:", dict(zip(FEATURE_COLUMNS, raw_test_sample.round(2).tolist())))
print("True species   :", true_species)
print()
for framework, predictor in [("Keras", keras_predict_flower), ("PyTorch", torch_predict_flower)]:
    label, confidence, probabilities = predictor(raw_test_sample)
    print(f"{framework:8s} -> {label} (confidence {confidence:.4f})")
    print("           probabilities:", dict(zip(CLASS_NAMES, probabilities.round(4).tolist())))

## 12. Optional: PyTensor, the maintained Theano lineage

Theano is discontinued. PyTensor and Aesara continue its symbolic-graph
approach. This section is optional: if PyTensor is not installed, it prints a
skip message and the rest of the notebook is unaffected.

To enable it locally:

```bash
pip install pytensor
```

In [ ]:
try:
    import pytensor
    import pytensor.tensor as pt

    X_sym = pt.matrix("X")
    W_shared = pytensor.shared(
        np.zeros((X_train.shape[1], len(CLASS_NAMES)), dtype=np.float32),
        name="W",
    )
    b_shared = pytensor.shared(
        np.zeros((len(CLASS_NAMES),), dtype=np.float32),
        name="b",
    )
    score_function = pytensor.function([X_sym], pt.dot(X_sym, W_shared) + b_shared)
    scores = score_function(X_test.astype(np.float32))
    print(f"PyTensor version: {pytensor.__version__}")
    print(f"Compiled symbolic graph output shape: {scores.shape}")
    print("Theano-style symbolic graphs still compile through the PyTensor fork.")
except Exception as exc:
    print(f"PyTensor demo skipped: {exc}")
    print("Install with: pip install pytensor")

## 13. Save models for later inference

Keras saves the full model. PyTorch saves a state dictionary; the architecture
is defined by `IrisNet`, so reloading requires that class.

In [ ]:
keras_path = OUTPUT_DIR / "exp02_iris_keras.keras"
torch_path = OUTPUT_DIR / "exp02_iris_torch.pt"

keras_model.save(keras_path)
torch.save(torch_model.state_dict(), torch_path)

print(f"Saved Keras model : {keras_path.resolve()}")
print(f"Saved PyTorch state: {torch_path.resolve()}")

reloaded_keras = tf.keras.models.load_model(keras_path)
reloaded_keras_predictions = reloaded_keras.predict(X_test, verbose=0).argmax(axis=1)
assert np.array_equal(reloaded_keras_predictions, keras_test_predictions)
print("Keras reload check passed.")

reloaded_torch = IrisNet(X_train.shape[1], len(CLASS_NAMES)).to(DEVICE)
reloaded_torch.load_state_dict(torch.load(torch_path, map_location=DEVICE))
reloaded_torch.eval()
with torch.no_grad():
    reloaded_torch_predictions = (
        reloaded_torch(torch.tensor(X_test, dtype=torch.float32).to(DEVICE))
        .argmax(dim=1)
        .cpu()
        .numpy()
    )
assert np.array_equal(reloaded_torch_predictions, torch_test_predictions)
print("PyTorch reload check passed.")

## 14. Conclusion

All four libraries solve the same mathematical problem with different levels
of abstraction:

- **Theano** established symbolic graphs and GPU acceleration but is no
  longer maintained. It is studied for historical context.
- **TensorFlow** exposes eager execution and `GradientTape` for custom
  training, while `tf.keras` provides the concise high-level path used here.
- **Keras** is the most compact way to express the model; Keras 3 can run on
  several backends.
- **PyTorch** makes the training loop explicit and remains the most flexible
  choice for research and custom architectures.

The two implementations in this notebook should both reach high accuracy on
Iris. Exact numbers depend on the random initialization, early-stopping
point, and framework version; the comparison table records whatever the run
produces.

**Artifacts created at runtime (not committed):**

- `outputs/exp02_iris_keras.keras`
- `outputs/exp02_iris_torch.pt`
- `outputs/exp02_framework_results.csv`
- `outputs/exp02_iris_scatter.png`
- `outputs/exp02_framework_training_curves.png`

**Execution note:** this notebook was syntax-checked and its model
architectures were reviewed during repository preparation, but the full
training run was not performed here. The lab sheet is conceptual, so there is
no single reference accuracy to reproduce.